In [1]:
"""
Expense Forecasting: AI Finance Tracker
Forecasts next month's spend per category using moving average and linear 
regression, evaluated with MAE and RMSE.
"""

"\nExpense Forecasting: AI Finance Tracker\nForecasts next month's spend per category using moving average and linear \nregression, evaluated with MAE and RMSE.\n"

In [2]:
from pathlib import Path

In [3]:
import pandas as pd
import numpy as np
import matplotlib

In [4]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [5]:
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [6]:
if "__file__" in globals():
    base_candidates = [Path(__file__).resolve().parent.parent]
else:
    cwd = Path.cwd()
    base_candidates = [cwd, cwd / "analytics", cwd.parent, cwd.parent / "analytics"]

In [7]:
BASE_DIR = None
for candidate in base_candidates:
    if (candidate / "data" / "sample_transactions.csv").exists():
        BASE_DIR = candidate
        break

In [8]:
if BASE_DIR is None:
    BASE_DIR = Path.cwd()

In [9]:
DATA_PATH = BASE_DIR / "data" / "sample_transactions.csv"
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
# ── Load and prepare data ─────────────────────────────
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df["month"] = df["date"].dt.to_period("M")

In [11]:
expenses = df[df["type"] == "expense"]
monthly_category = expenses.groupby(["month", "category"])["amount"].sum().unstack(fill_value=0)
monthly_category.index = monthly_category.index.astype(str)

In [12]:
print(f"Monthly category spend matrix: {monthly_category.shape[0]} months x {monthly_category.shape[1]} categories")
monthly_category.head()

Monthly category spend matrix: 12 months x 10 categories


category,Dining,Entertainment,Groceries,Healthcare,Investments,Rent,Shopping,Subscriptions,Transport,Utilities
month,,,,,,,,,,
2025-07,28869.66,2638.29,23445.61,3241.09,7603.65,24180.00,1394.60,2401.09,11601.23,4346.66
2025-08,10559.90,0.00,25305.66,1870.65,8658.77,12362.70,2503.49,460.79,9274.87,2295.47
2025-09,8312.92,1140.97,26594.85,971.38,10424.13,12548.14,2522.65,896.83,17071.13,2201.85
2025-10,10656.23,2968.67,25530.30,566.67,9983.30,12736.36,1275.55,1559.22,10983.18,2519.43
2025-11,4410.71,2643.12,22608.26,0.00,4742.83,12927.41,8589.55,672.27,8168.53,1802.80


In [13]:
# ── Method 1: Moving average forecast (3-month window) ──
ma_forecast = monthly_category.rolling(window=3, min_periods=1).mean().iloc[-1]

In [14]:
print("Moving Average Forecast (next month, per category):")
print(ma_forecast.round(0).sort_values(ascending=False))

Moving Average Forecast (next month, per category):
category
Groceries        26338.0
Transport        10965.0
Dining            9879.0
Rent              9354.0
Shopping          6438.0
Investments       4760.0
Healthcare        3607.0
Entertainment     2189.0
Utilities         1266.0
Subscriptions      384.0
Name: 2026-06, dtype: float64


In [15]:
# ── Method 2: Linear regression forecast per category ──
results = {}
forecast_next = {}

In [16]:
for category in monthly_category.columns:
    y = monthly_category[category].values
    X = np.arange(len(y)).reshape(-1, 1)

    if len(y) < 3:
        continue

    # Train on all but last month, test on last month
    X_train, y_train = X[:-1], y[:-1]
    X_test, y_test = X[-1:], y[-1:]

    model = LinearRegression()
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    # Forecast next month (one step beyond all available data)
    next_month_pred = model.predict([[len(y)]])[0]

    results[category] = {"MAE": round(mae, 2), "RMSE": round(rmse, 2)}
    forecast_next[category] = round(max(next_month_pred, 0), 2)

In [17]:
results_df = pd.DataFrame(results).T.sort_values("RMSE", ascending=False)
print("\nLinear Regression Model Accuracy (last month held out):")
print(results_df)


Linear Regression Model Accuracy (last month held out):
                    MAE      RMSE
Rent           12126.97  12126.97
Groceries      10631.22  10631.22
Transport       8245.10   8245.10
Dining          7313.35   7313.35
Investments     6239.46   6239.46
Shopping        5001.14   5001.14
Healthcare      2837.46   2837.46
Utilities       1804.20   1804.20
Subscriptions    546.70    546.70
Entertainment    234.98    234.98


In [18]:
print(f"\nAverage MAE across categories: ₹{results_df['MAE'].mean():,.2f}")
print(f"Average RMSE across categories: ₹{results_df['RMSE'].mean():,.2f}")


Average MAE across categories: ₹5,498.06
Average RMSE across categories: ₹5,498.06


In [19]:
forecast_df = pd.Series(forecast_next).sort_values(ascending=False)
print("\nLinear Regression Forecast — Next Month Spend by Category:")
print(forecast_df)


Linear Regression Forecast — Next Month Spend by Category:
Groceries        29326.25
Transport        16156.90
Rent             11777.00
Investments       5978.14
Entertainment     3332.45
Dining            3192.28
Shopping          3148.17
Healthcare        2132.96
Utilities         1697.91
Subscriptions      465.88
dtype: float64


In [20]:
# ── Visualize actual vs predicted for top 3 categories ──
top_categories = monthly_category.sum().sort_values(ascending=False).head(3).index

In [21]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, category in zip(axes, top_categories):
    y = monthly_category[category].values
    months = monthly_category.index

    ax.plot(months, y, marker="o", label="Actual")
    ax.axhline(ma_forecast[category], color="orange", linestyle="--", label="MA Forecast")
    ax.axhline(forecast_next[category], color="red", linestyle="--", label="LR Forecast")
    ax.set_title(category)
    ax.tick_params(axis="x", rotation=45)
    ax.legend(fontsize=8)

In [22]:
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "forecast_comparison.png", dpi=120)
plt.show()

C:\Users\home\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:3: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  This is separate from the ipykernel package so we can avoid doing imports until


In [23]:
# ── Combined forecast summary ─────────────────────────
summary = pd.DataFrame({
    "Moving_Avg_Forecast": ma_forecast.round(0),
    "Linear_Reg_Forecast": forecast_df.round(0),
}).sort_values("Linear_Reg_Forecast", ascending=False)

In [24]:
print("\nForecast Summary — Next Month:")
print(summary)


Forecast Summary — Next Month:
               Moving_Avg_Forecast  Linear_Reg_Forecast
Groceries                  26338.0              29326.0
Transport                  10965.0              16157.0
Rent                        9354.0              11777.0
Investments                 4760.0               5978.0
Entertainment               2189.0               3332.0
Dining                      9879.0               3192.0
Shopping                    6438.0               3148.0
Healthcare                  3607.0               2133.0
Utilities                   1266.0               1698.0
Subscriptions                384.0                466.0


In [25]:
highest_forecast_cat = summary["Linear_Reg_Forecast"].idxmax()
print(f"\nInsight: {highest_forecast_cat} is forecasted to be the highest expense "
      f"category next month at ₹{summary['Linear_Reg_Forecast'].max():,.0f}, "
      f"based on linear regression trend.")


Insight: Groceries is forecasted to be the highest expense category next month at ₹29,326, based on linear regression trend.


In [26]:
print("\n✅ Forecasting complete. Chart saved to analytics/outputs/")


✅ Forecasting complete. Chart saved to analytics/outputs/
